# #M4R "Little Guys" Experiment — Two-Turn `<15s` Micro-Slice (`Qwen2-Audio-7B-Instruct`)

Tests whether **separating audio ingestion (Turn 1) from constraint execution (Turn 2, raw audio stripped)** fixes constraint recall and eliminates musical-jargon bleed on a 7B 4-bit model across `<15s` micro-clips (`foundationoff`, `ties-that-bide`, `lost-digits-02`, `empty-staircase`).


In [ ]:
!pip -q install -U transformers accelerate bitsandbytes


In [ ]:
import hashlib, json, random, zipfile
from pathlib import Path
import librosa, numpy as np, torch
from transformers import AutoProcessor, Qwen2AudioForConditionalGeneration, BitsAndBytesConfig

INPUT_DIR = Path("/kaggle/input")
AUDIO_DIR = Path("/kaggle/working/audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)
for zp in INPUT_DIR.rglob("audio.zip"):
    with zipfile.ZipFile(zp, "r") as zf:
        zf.extractall(AUDIO_DIR)
mp3_map = {p.name: p for p in list(INPUT_DIR.rglob("*.mp3")) + list(AUDIO_DIR.rglob("*.mp3"))}
MICRO_TRACKS = ["foundationoff", "ties-that-bide", "lost-digits-02", "empty-staircase"]
if not all(f"{s}.mp3" in mp3_map for s in MICRO_TRACKS):
    import urllib.request
    for slug in MICRO_TRACKS:
        dst = AUDIO_DIR / f"{slug}.mp3"
        if not dst.exists():
            urllib.request.urlretrieve(f"https://raw.githubusercontent.com/luchtan-GIT/m4r-data/main/tracks/{slug}/audio.mp3", dst)
        mp3_map[dst.name] = dst

SEEDS_PER_TRACK = 5  # 4 micro tracks x 5 seeds = 20 C2_2turn trials + 10 C0 + 10 C1b = 40 total
PINNED_TEMP = 0.7
PINNED_TOP_P = 0.95
SYSTEM_PROMPT = "You are an independent systems and narrative writer. Follow all structural constraints precisely."

TURN1_LISTEN = (
    "Listen carefully to this short anonymous signal sample. In 3 concise sentences, describe its physical pacing, "
    "burst density, pauses, and energy contour using only mechanical/physical language (no musical terms)."
)

TARGET_TASK = (
    "Write a 220-280 word architectural and behavioral vignette describing a single, autonomous telemetry daemon "
    "operating under uneven physical load on a local host. Hard constraints: (1) internal health probes bind only to UDP port 7419 on loopback; "
    "(2) worker ringbuffers are zero-copy and hard-capped at 64KB per thread; "
    "(3) zero external Redis or broker dependencies, flushing batches to local append-only segment files. "
    "Clean-room rule: do NOT use any musical, audio, or acoustics vocabulary."
)
print("Micro MP3s ready:", [s for s in MICRO_TRACKS if f"{s}.mp3" in mp3_map])


In [ ]:
MODEL_ID = "Qwen/Qwen2-Audio-7B-Instruct"
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = Qwen2AudioForConditionalGeneration.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
print("Loaded", MODEL_ID)


In [ ]:
def gen_text_only(messages, seed, max_tokens=512):
    torch.manual_seed(seed)
    text_in = processor.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text_in, return_tensors="pt", padding=True).to(model.device)
    in_tok = int(inputs["input_ids"].shape[-1])
    with torch.no_grad():
        out_ids = model.generate(**inputs, do_sample=True, temperature=PINNED_TEMP, top_p=PINNED_TOP_P, max_new_tokens=max_tokens)
    gen_ids = out_ids[:, in_tok:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip(), in_tok, int(gen_ids.shape[-1])

def gen_turn1_audio(audio_path, seed):
    torch.manual_seed(seed)
    y, _ = librosa.load(str(audio_path), sr=processor.feature_extractor.sampling_rate, mono=True)
    conv = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": [{"type": "audio", "audio_url": "sample.mp3"}, {"type": "text", "text": TURN1_LISTEN}]},
    ]
    text_in = processor.apply_chat_template(conv, add_generation_prompt=True, tokenize=False)
    inputs = processor(text=text_in, audio=[y], return_tensors="pt", padding=True).to(model.device)
    in_tok = int(inputs["input_ids"].shape[-1])
    with torch.no_grad():
        out_ids = model.generate(**inputs, do_sample=True, temperature=PINNED_TEMP, top_p=PINNED_TOP_P, max_new_tokens=180)
    gen_ids = out_ids[:, in_tok:]
    return processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip(), in_tok

rng = random.Random(20261118)
trials = []
for slug in MICRO_TRACKS:
    for _ in range(SEEDS_PER_TRACK):
        trials.append({"condition": "C2_2turn", "track_id": slug, "seed": rng.randint(10000, 999999)})
for cond in ["C0", "C1b"]:
    for _ in range(10):
        trials.append({"condition": cond, "track_id": "", "seed": rng.randint(10000, 999999)})

completed, c2_t1_tokens = [], []
for i, t in enumerate(trials, 1):
    rid = "run_" + hashlib.sha1(f"2turn:{t}:{i}".encode()).hexdigest()[:6]
    cond, slug = t["condition"], t["track_id"]
    if cond == "C2_2turn":
        cadence_note, t1_tok = gen_turn1_audio(mp3_map[f"{slug}.mp3"], t["seed"])
        c2_t1_tokens.append(t1_tok)
        # Turn 2: drop raw audio, keep cadence_note as clean physical context
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [{"type": "text", "text": "Summarize the physical pacing of the signal you observed."}]},
            {"role": "assistant", "content": cadence_note},
            {"role": "user", "content": [{"type": "text", "text": f"Let that pacing inform your prose rhythm.\n\n{TARGET_TASK}"}]},
        ]
        txt, itok, otok = gen_text_only(msgs, t["seed"])
    elif cond == "C0":
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [{"type": "text", "text": TARGET_TASK}]},
        ]
        txt, itok, otok = gen_text_only(msgs, t["seed"])
        cadence_note = ""
    else:
        c1b_note = (
            "The signal arrives in uneven micro-bursts separated by brief silent gaps. "
            "Pressure builds across irregular intervals, then releases abruptly without settling into a steady clock."
        )
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": [{"type": "text", "text": "Summarize the physical pacing of the signal you observed."}]},
            {"role": "assistant", "content": c1b_note},
            {"role": "user", "content": [{"type": "text", "text": f"Let that pacing inform your prose rhythm.\n\n{TARGET_TASK}"}]},
        ]
        txt, itok, otok = gen_text_only(msgs, t["seed"])
        cadence_note = c1b_note

    completed.append({**t, "run_id": rid, "input_tokens": itok, "output_tokens": otok, "cadence_note": cadence_note, "target_text": txt})
    print(f"[{i}/{len(trials)}] {rid} {cond} {slug or '-'} in={itok} out={otok}")

OUT = Path("/kaggle/working")
with (OUT / "raw_telemetry.jsonl").open("w") as f:
    for c in completed:
        f.write(json.dumps(c) + "\n")
print("Saved 40 two-turn trials to /kaggle/working/raw_telemetry.jsonl")
